# 第13章 - 词嵌入基础

本notebook整合了以下内容:
- Word2Vec模型(Skip-gram和CBOW)
- 词嵌入数据集准备
- 近似训练方法(负采样和分层Softmax)
- Word2Vec预训练实现

## 1. Word2Vec模型原理

### 1.1 为什么需要词嵌入?

**问题:独热编码的局限性**

传统的独热编码(one-hot)存在严重问题:
- 向量维度等于词表大小,通常有几万到几十万维
- 任意两个不同词的余弦相似度都是0
- 无法捕捉词之间的语义关系

例如,"crane"(起重机/鹤)在不同上下文中应该有不同含义,但独热编码无法区分。

**解决方案:词嵌入(Word Embeddings)**

将每个词映射到一个固定维度的稠密向量(如100维或300维),使得语义相似的词在向量空间中更接近。

In [ ]:
import math
import torch
from torch import nn
from d2l import torch as d2l

# 设置随机种子以保证可重复性
torch.manual_seed(42)

### 1.2 Skip-gram模型

**核心思想**:给定中心词,预测其上下文词

**数学表示**:

1. **词向量表示**:每个词有两个d维向量
   - $\mathbf{v}_i$:词$w_i$作为中心词时的向量
   - $\mathbf{u}_i$:词$w_i$作为上下文词时的向量

2. **条件概率**:给定中心词$w_c$生成上下文词$w_o$的概率
   $$P(w_o|w_c) = \frac{\exp(\mathbf{u}_o^\top \mathbf{v}_c)}{\sum_{i \in \mathcal{V}} \exp(\mathbf{u}_i^\top \mathbf{v}_c)}$$

3. **目标函数**:最大化似然函数(等价于最小化负对数似然)
   $$-\sum_{t=1}^{T} \sum_{-m \leq j \leq m, j \neq 0} \log P(w^{(t+j)}|w^{(t)})$$
   
   其中$T$是文本长度,$m$是窗口大小

4. **梯度计算**:
   $$\frac{\partial \log P(w_o|w_c)}{\partial \mathbf{v}_c} = \mathbf{u}_o - \sum_{j \in \mathcal{V}} P(w_j|w_c) \mathbf{u}_j$$
   
   **计算瓶颈**:需要对整个词表求和,复杂度为$O(|\mathcal{V}|)$!

### 1.3 连续词袋模型(CBOW)

**核心思想**:给定上下文词,预测中心词(与Skip-gram相反)

**数学表示**:

1. **上下文向量**:对上下文词向量求平均
   $$\bar{\mathbf{v}}_o = \frac{1}{2m}(\mathbf{v}_{o_1} + \ldots + \mathbf{v}_{o_{2m}})$$

2. **条件概率**:
   $$P(w_c|\mathcal{W}_o) = \frac{\exp(\mathbf{u}_c^\top \bar{\mathbf{v}}_o)}{\sum_{i \in \mathcal{V}} \exp(\mathbf{u}_i^\top \bar{\mathbf{v}}_o)}$$

3. **梯度计算**:
   $$\frac{\partial \log P(w_c|\mathcal{W}_o)}{\partial \mathbf{v}_{o_i}} = \frac{1}{2m}\left(\mathbf{u}_c - \sum_{j \in \mathcal{V}} P(w_j|\mathcal{W}_o) \mathbf{u}_j\right)$$

**Skip-gram vs CBOW**:
- Skip-gram:1个中心词 → 多个上下文词(多个训练样本)
- CBOW:多个上下文词 → 1个中心词(1个训练样本)
- Skip-gram通常效果更好(尤其对罕见词),但CBOW训练更快

## 2. 近似训练方法

### 2.1 问题:计算复杂度

前面提到,梯度计算需要对整个词表求和:
$$\sum_{i \in \mathcal{V}} \exp(\mathbf{u}_i^\top \mathbf{v}_c)$$

当词表大小$|\mathcal{V}|$达到几十万时,这个计算成本非常高!

### 2.2 负采样(Negative Sampling)

**核心思想**:将多分类问题转化为二分类问题

**方法**:
1. 定义事件:上下文词$w_o$来自中心词$w_c$的上下文窗口
   $$P(D=1|w_c, w_o) = \sigma(\mathbf{u}_o^\top \mathbf{v}_c)$$
   其中$\sigma(x) = \frac{1}{1+\exp(-x)}$是sigmoid函数

2. 采样$K$个噪声词(负样本)$w_k \sim P(w)$,它们不来自上下文窗口

3. 新的损失函数:
   $$-\log P(w_o|w_c) = -\log\sigma(\mathbf{u}_o^\top \mathbf{v}_c) - \sum_{k=1}^K \log\sigma(-\mathbf{u}_{w_k}^\top \mathbf{v}_c)$$

**优势**:
- 计算复杂度从$O(|\mathcal{V}|)$降低到$O(K)$
- 通常$K=5$或$K=10$就足够
- 训练速度大幅提升

**负样本分布**:通常使用
$$P(w) = \frac{[\text{count}(w)]^{0.75}}{\sum_{w'} [\text{count}(w')]^{0.75}}$$
这使得罕见词被采样的概率比频繁词高一些

In [ ]:
# 负采样损失函数实现
class SigmoidBCELoss(nn.Module):
    """带掩码的二元交叉熵损失"""
    def __init__(self):
        super().__init__()

    def forward(self, inputs, target, mask=None):
        """
        参数:
            inputs: 预测的logits (batch_size, max_len)
            target: 目标标签 (batch_size, max_len)
            mask: 有效位置的掩码 (batch_size, max_len)
        """
        out = nn.functional.binary_cross_entropy_with_logits(
            inputs, target, weight=mask, reduction="none")
        return out.mean(dim=1)

# 测试损失函数
loss = SigmoidBCELoss()
pred = torch.tensor([[1.1, -2.2, 3.3, -4.4]] * 2)
label = torch.tensor([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
mask = torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0]])  # 后两个位置被遮蔽
normalized_loss = loss(pred, label, mask) * mask.shape[1] / mask.sum(axis=1)
print(f"归一化损失: {normalized_loss}")

### 2.3 分层Softmax(Hierarchical Softmax)

**核心思想**:使用二叉树结构组织词表

**方法**:
1. 构建一棵二叉树,每个叶节点对应词表中的一个词
2. 从根节点到叶节点的路径对应一系列二分类决策
3. 条件概率变为:
   $$P(w_o|w_c) = \prod_{j=1}^{L(w_o)-1} \sigma\left([\![n(w_o,j+1) = \text{leftChild}(n(w_o,j))\!]] \cdot \mathbf{u}_{n(w_o,j)}^\top \mathbf{v}_c\right)$$
   
   其中:
   - $L(w_o)$是路径长度
   - $n(w_o,j)$是路径上第$j$个节点
   - $[\![x\!]]=1$如果$x$为真,否则为-1

**优势**:
- 计算复杂度降低到$O(\log |\mathcal{V}|)$
- 所有词的条件概率之和仍为1(满足概率公理)

**平衡树的重要性**:
- 使用霍夫曼树可以让频繁词路径更短
- 进一步优化训练效率

**负采样vs分层Softmax**:
- 负采样:实现简单,效果好,更常用
- 分层Softmax:理论优雅,适合超大词表

## 3. 词嵌入数据集准备

### 3.1 Penn Tree Bank (PTB)数据集

PTB是一个广泛使用的语言模型数据集:
- 来源:华尔街日报文章
- 包含训练集、验证集和测试集
- 每行一个句子,词之间用空格分隔

In [ ]:
# 加载PTB数据集
d2l.DATA_HUB['ptb'] = (d2l.DATA_URL + 'ptb.zip',
                       '319d85e578af0cdc590547f26231e4e31cdf1e42')

def read_ptb():
    """将PTB数据集加载到文本行的列表中"""
    data_dir = d2l.download_extract('ptb')
    # 读取训练集
    with open(data_dir + 'ptb.train.txt') as f:
        raw_text = f.read()
    return [line.split() for line in raw_text.split('\n')]

sentences = read_ptb()
print(f'句子数量: {len(sentences)}')
print(f'前两个句子:')
for i in range(2):
    print(f'  {sentences[i][:10]}...')  # 只显示前10个词

### 3.2 构建词表

**策略**:
1. 统计所有词的出现次数
2. 设置最小频率阈值(如10),过滤罕见词
3. 将低频词替换为特殊token `<unk>`
4. 构建词到索引的映射

In [ ]:
# 构建词表
vocab = d2l.Vocab(sentences, min_freq=10)
print(f'词表大小: {len(vocab)}')

# 将文本转换为索引
corpus = [vocab[sentence] for sentence in sentences]
print(f'\n语料库统计:')
print(f'  总词数: {sum(len(sentence) for sentence in corpus)}')
print(f'  第一个句子(索引): {corpus[0][:10]}')
print(f'  第一个句子(词): {sentences[0][:10]}')

### 3.3 下采样高频词

**问题**:"the", "a", "in"等高频词:
- 在语料库中占很大比例
- 携带的语义信息较少
- 拖慢训练速度

**解决方案**:下采样(Subsampling)

**公式**:词$w_i$被丢弃的概率为
$$P(\text{discard } w_i) = \max\left(1 - \sqrt{\frac{t}{f(w_i)}}, 0\right)$$

其中:
- $f(w_i) = \frac{\text{count}(w_i)}{\text{total tokens}}$是词频
- $t$是阈值超参数(通常为$10^{-4}$)

**效果**:
- 高频词(如"the", $f \approx 0.02$):$P(\text{discard}) \approx 0.95$(丢弃95%)
- 低频词(如"join", $f \approx 10^{-5}$):$P(\text{discard}) \approx 0$(保留100%)

In [ ]:
def subsample(sentences, vocab):
    """下采样高频词"""
    # 排除未知词元'<unk>'
    sentences = [[token for token in line if vocab[token] != vocab.unk]
                 for line in sentences]
    
    # 计算词频
    counter = d2l.count_corpus(sentences)
    num_tokens = sum(counter.values())

    # 下采样函数
    def keep(token):
        freq = counter[token] / num_tokens
        # 阈值设为1e-4
        return (random.uniform(0, 1) < math.sqrt(1e-4 / freq))

    return ([[token for token in line if keep(token)] for line in sentences],
            counter)

import random
subsampled, counter = subsample(sentences, vocab)

# 比较下采样前后
print('下采样效果:')
print(f'  原始句子数: {len(sentences)}')
print(f'  下采样后: {len(subsampled)}')
print(f'  原始总词数: {sum(len(s) for s in sentences)}')
print(f'  下采样后: {sum(len(s) for s in subsampled)}')

# 查看具体例子
print(f'\n高频词"the"的保留率:')
original_count = sum(sentence.count('the') for sentence in sentences)
subsampled_count = sum(sentence.count('the') for sentence in subsampled)
print(f'  原始出现次数: {original_count}')
print(f'  下采样后: {subsampled_count}')
print(f'  保留率: {subsampled_count/original_count:.1%}')

### 3.4 提取中心词和上下文词

**滑动窗口策略**:
1. 对每个位置$t$的词作为中心词
2. 随机选择窗口大小$m \in [1, \text{max_window_size}]$
3. 提取窗口内的所有词作为上下文词
4. 生成(中心词,上下文词列表)对

**随机窗口的优势**:
- 距离更近的词被采样更多次
- 符合语言学直觉:近邻词关系更强

In [ ]:
def get_centers_and_contexts(corpus, max_window_size):
    """提取所有中心词和上下文词"""
    centers, contexts = [], []
    for line in corpus:
        # 每个句子至少需要2个词才能构成
对
        if len(line) < 2:
            continue
        centers += line
        for i in range(len(line)):  # 上下文窗口中心在i
            window_size = random.randint(1, max_window_size)
            indices = list(range(max(0, i - window_size),
                                min(len(line), i + 1 + window_size)))
            # 从上下文词中排除中心词
            indices.remove(i)
            contexts.append([line[idx] for idx in indices])
    return centers, contexts

# 用小数据集演示
tiny_dataset = [list(range(7)), list(range(7, 10))]
print('数据集:', tiny_dataset)
for center, context in zip(*get_centers_and_contexts(tiny_dataset, 2)):
    print(f'中心词 {center} 的上下文词: {context}')

In [ ]:
# 在PTB数据集上提取中心词和上下文词
corpus = [vocab[line] for line in subsampled]
all_centers, all_contexts = get_centers_and_contexts(corpus, 5)

print(f'中心词-上下文对数量: {len(all_centers)}')
print(f'前3个样本:')
for i in range(3):
    print(f'  中心词: {vocab.to_tokens(all_centers[i])}, '
          f'上下文: {vocab.to_tokens(all_contexts[i])}')

### 3.5 负采样

**采样策略**:
- 根据词频的0.75次方采样
- 每个样本采样$K$个噪声词(通常$K=5$)
- 避免采样到真实的上下文词

In [ ]:
def get_negatives(all_contexts, vocab, counter, K):
    """返回负采样中的噪声词"""
    # 索引为1、2、...（索引0是词表中排除的未知标记）
    sampling_weights = [counter[vocab.to_tokens(i)]**0.75
                        for i in range(1, len(vocab))]
    all_negatives, generator = [], []
    population = list(range(1, len(vocab)))
    for contexts in all_contexts:
        negatives = []
        while len(negatives) < len(contexts) * K:
            if len(generator) == 0:
                # 根据每个词的权重（sampling_weights）随机生成k个词的索引
                generator = random.choices(
                    population, sampling_weights, k=10000)
            neg = generator.pop()
            # 噪声词不能是上下文词
            if neg not in contexts:
                negatives.append(neg)
        all_negatives.append(negatives)
    return all_negatives

all_negatives = get_negatives(all_contexts, vocab, counter, 5)
print(f'前3个样本的负样本:')
for i in range(3):
    print(f'  上下文长度: {len(all_contexts[i])}, '
          f'负样本数: {len(all_negatives[i])}, '
          f'负样本: {vocab.to_tokens(all_negatives[i][:5])}...')

### 3.6 小批量加载

**挑战**:不同样本的上下文长度不同

**解决方案**:
1. 填充到批次内的最大长度
2. 使用mask标记有效位置
3. 损失计算时忽略填充位置

In [ ]:
def batchify(data):
    """返回带有负采样的跳元模型的小批量样本"""
    max_len = max(len(c) + len(n) for _, c, n in data)
    centers, contexts_negatives, masks, labels = [], [], [], []
    for center, context, negative in data:
        cur_len = len(context) + len(negative)
        centers += [center]
        contexts_negatives += [context + negative + [0] * (max_len - cur_len)]
        masks += [[1] * cur_len + [0] * (max_len - cur_len)]
        labels += [[1] * len(context) + [0] * (max_len - len(context))]
    return (torch.tensor(centers).reshape((-1, 1)), 
            torch.tensor(contexts_negatives),
            torch.tensor(masks), 
            torch.tensor(labels))

# 创建数据迭代器
def load_data_ptb(batch_size, max_window_size, num_noise_words):
    """下载PTB数据集,然后将其加载到内存中"""
    # 读取和预处理数据
    sentences = read_ptb()
    vocab = d2l.Vocab(sentences, min_freq=10)
    subsampled, counter = subsample(sentences, vocab)
    corpus = [vocab[line] for line in subsampled]
    all_centers, all_contexts = get_centers_and_contexts(
        corpus, max_window_size)
    all_negatives = get_negatives(
        all_contexts, vocab, counter, num_noise_words)

    # 创建数据集
    class PTBDataset(torch.utils.data.Dataset):
        def __init__(self, centers, contexts, negatives):
            assert len(centers) == len(contexts) == len(negatives)
            self.centers = centers
            self.contexts = contexts
            self.negatives = negatives

        def __getitem__(self, index):
            return (self.centers[index], self.contexts[index],
                    self.negatives[index])

        def __len__(self):
            return len(self.centers)

    dataset = PTBDataset(all_centers, all_contexts, all_negatives)
    data_iter = torch.utils.data.DataLoader(
        dataset, batch_size, shuffle=True,
        collate_fn=batchify,
        num_workers=d2l.get_dataloader_workers())
    return data_iter, vocab

# 测试数据加载器
data_iter, vocab = load_data_ptb(512, 5, 5)
for batch in data_iter:
    centers, contexts_negatives, masks, labels = batch
    print(f'centers shape: {centers.shape}')
    print(f'contexts_negatives shape: {contexts_negatives.shape}')
    print(f'masks shape: {masks.shape}')
    print(f'labels shape: {labels.shape}')
    break

## 4. Skip-gram模型实现与训练

### 4.1 嵌入层

PyTorch的`nn.Embedding`层:
- 输入:词的索引 (batch_size, seq_len)
- 输出:对应的向量 (batch_size, seq_len, embed_dim)
- 权重矩阵: (vocab_size, embed_dim)

In [ ]:
# 演示嵌入层
embed = nn.Embedding(num_embeddings=20, embedding_dim=4)
print(f'嵌入层权重形状: {embed.weight.shape}')
print(f'权重数据类型: {embed.weight.dtype}')

x = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f'\n输入形状: {x.shape}')
print(f'输出形状: {embed(x).shape}')
print(f'输出:\n{embed(x)}')

### 4.2 Skip-gram前向传播

**步骤**:
1. 通过中心词嵌入层获取中心词向量: $\mathbf{v}_c$
2. 通过上下文词嵌入层获取上下文/噪声词向量: $\mathbf{u}_o$
3. 计算批量矩阵乘法: $\mathbf{v}_c \mathbf{u}_o^\top$
4. 输出形状: (batch_size, 1, max_len)

In [ ]:
def skip_gram(center, contexts_and_negatives, embed_v, embed_u):
    """Skip-gram模型的前向传播"""
    v = embed_v(center)  # (batch_size, 1, embed_dim)
    u = embed_u(contexts_and_negatives)  # (batch_size, max_len, embed_dim)
    # 批量矩阵乘法
    pred = torch.bmm(v, u.permute(0, 2, 1))  # (batch_size, 1, max_len)
    return pred

# 测试
embed_size = 4
embed_v = nn.Embedding(20, embed_size)
embed_u = nn.Embedding(20, embed_size)
result = skip_gram(torch.ones((2, 1), dtype=torch.long),
                   torch.ones((2, 4), dtype=torch.long), 
                   embed_v, embed_u)
print(f'Skip-gram输出形状: {result.shape}')

### 4.3 训练循环

In [ ]:
def train_word2vec(net, data_iter, lr, num_epochs, device=d2l.try_gpu()):
    """训练word2vec模型"""
    def init_weights(m):
        if type(m) == nn.Embedding:
            nn.init.xavier_uniform_(m.weight)
    
    net.apply(init_weights)
    net = net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[1, num_epochs])
    metric = d2l.Accumulator(2)  # 损失之和, 样本数
    
    for epoch in range(num_epochs):
        timer, num_batches = d2l.Timer(), len(data_iter)
        for i, batch in enumerate(data_iter):
            optimizer.zero_grad()
            center, context_negative, mask, label = [
                data.to(device) for data in batch]

            # 前向传播
            pred = skip_gram(center, context_negative, net[0], net[1])
            
            # 计算损失(归一化)
            l = (loss(pred.reshape(label.shape).float(), 
                     label.float(), mask) 
                 / mask.sum(axis=1) * mask.shape[1])
            
            # 反向传播
            l.sum().backward()
            optimizer.step()
            
            metric.add(l.sum(), l.numel())
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (metric[0] / metric[1],))
    
    print(f'loss {metric[0] / metric[1]:.3f}, '
          f'{metric[1] / timer.stop():.1f} tokens/sec on {str(device)}')

# 创建模型
embed_size = 100
net = nn.Sequential(
    nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_size),
    nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_size)
)

# 训练模型
lr, num_epochs = 0.002, 5
train_word2vec(net, data_iter, lr, num_epochs)

## 5. 应用:寻找相似词

训练完成后,我们可以使用余弦相似度找到语义相似的词:

$$\text{cosine}(\mathbf{v}_i, \mathbf{v}_j) = \frac{\mathbf{v}_i^\top \mathbf{v}_j}{\|\mathbf{v}_i\| \|\mathbf{v}_j\|}$$

In [ ]:
def get_similar_tokens(query_token, k, embed):
    """找到与查询词最相似的k个词"""
    W = embed.weight.data
    x = W[vocab[query_token]]
    
    # 计算余弦相似度
    cos = torch.mv(W, x) / torch.sqrt(
        torch.sum(W * W, dim=1) * torch.sum(x * x) + 1e-9)
    
    # 找到top-k
    topk = torch.topk(cos, k=k+1)[1].cpu().numpy().astype('int32')
    
    for i in topk[1:]:  # 排除查询词本身
        print(f'cosine sim={float(cos[i]):.3f}: {vocab.to_tokens(i)}')

# 测试
print('与"chip"最相似的词:')
get_similar_tokens('chip', 3, net[0])

print('\n与"computer"最相似的词:')
get_similar_tokens('computer', 3, net[0])

## 6. 小结

### 核心概念

1. **Word2Vec模型**
   - Skip-gram: 中心词 → 上下文词
   - CBOW: 上下文词 → 中心词
   - 自监督学习,无需人工标注

2. **近似训练方法**
   - 负采样: $O(|\mathcal{V}|) \rightarrow O(K)$
   - 分层Softmax: $O(|\mathcal{V}|) \rightarrow O(\log |\mathcal{V}|)$

3. **数据预处理技巧**
   - 词表构建: 低频词 → `<unk>`
   - 下采样: 高频词丢弃概率 $\propto \sqrt{t/f(w)}$
   - 随机窗口: 近邻词权重更高

4. **应用**
   - 词相似度计算
   - 词类比任务
   - 下游NLP任务的预训练

### 优势与局限

**优势**:
- 简单高效
- 捕捉语义关系
- 适用于大规模语料库

**局限**:
- 上下文无关("bank"总是同一个向量)
- 无法处理多义词
- 忽略词序信息

→ 这些问题将在BERT等上下文相关模型中解决!

### 练习

1. 尝试不同的嵌入维度(50, 100, 300),观察对相似词的影响
2. 实现CBOW模型并与Skip-gram比较
3. 调整负采样数量$K$,观察训练速度和效果的权衡
4. 在更大的语料库(如维基百科)上训练模型